In [ ]:
!pip install -U "bitsandbytes>=0.42.0"

In [ ]:
# 1. Install Unsloth first
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Install the helper libraries WITHOUT version constraints
# This prevents the 'building wheel' error by letting pip find the binary versions
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-34grf5jn/unsloth_865885ec78d94379a9254f6b005a7413
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-34grf5jn/unsloth_865885ec78d94379a9254f6b005a7413
  Resolved https://github.com/unslothai/unsloth.git to commit e51d3ea2e498fc893770d92ca6727bd113918480
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from unsloth import FastLanguageModel
import torch

# 1. Configuration
max_seq_length = 2048
dtype = None # None for auto detection. Float16 for Tesla T4
load_in_4bit = True # Use 4bit quantization to save memory

# 2. Load Llama 3.2 3B
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 3. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Increased slightly to capture personality nuances
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32, # Always keep alpha = r or 2*r
    lora_dropout = 0.05, # Small dropout to help it generalize sensor data
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("Model loaded successfully! Ready for your dataset.")

==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model loaded successfully! Ready for your dataset.


In [ ]:
from datasets import load_dataset

# 1. Load the JSON file
dataset = load_dataset("json", data_files="perdata.json", split="train")

# 2. Define the Prompt Template (Llama 3.2 format)
prompt_style = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a precise squat coach AI. Analyze sensor data (Spine, Knee, Ankle, Openness, Conscientiousness, Agreeableness, Extraversion, Neuroticism ) and provide concise, prioritized cues.<|eot_id|><|start_header_id|>user<|end_header_id|>

Instruction: {instruction}
Input Data: {input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{output}<|eot_id|>"""

# 3. Format function
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = prompt_style.format(instruction=instruction, input=input_text, output=output)
        texts.append(text)
    return { "text" : texts, }

# 4. Map the dataset
dataset = dataset.map(formatting_prompts_func, batched = True)

print(f"Dataset prepared! Total examples: {len(dataset)}")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,

        # FIX 1: Increase steps to 300 so it sees the data ~4 times
        max_steps = 300,

        # FIX 2: Lower learning rate for more "stable" personality learning
        learning_rate = 1e-4,

        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",

        # FIX 3: Higher weight decay stops the model from repeating "Excellent work!"
        weight_decay = 0.05,

        # FIX 4: Use Cosine scheduler for a smoother finish
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

In [ ]:
from transformers import TextStreamer

# 1. Format the tokenizer for Chat
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model)

# 2. Define Input
instruction = "You are a concise squat coach. Given spine, knee, and ankle conditions (0-1), Big Five personality traits (0-1), and a list of issues, identify the priority area and give 1-2 short cues. Max 18 words."
input_data = "Spine: 0.3, Knee: 0.9, Ankle: 0.9. Openness: 0.68, Conscientiousness: 0.7, Agreeableness: 0.7, Extraversion: 0.9, Neuroticism: 0.7. Issues:Bend forward, Shallow squat, knee crossing toe."

# 3. Format Messages
messages = [
    {"role": "system", "content": instruction},
    {"role": "user", "content": input_data},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# 4. Initialize Streamer AND Generate
# This line was missing or misplaced!
text_streamer = TextStreamer(tokenizer)

print("--- SQUAT COACH RESPONSE ---")
_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 64,
    use_cache = True,
    temperature = 0.3,
    top_p = 0.9,
    repetition_penalty = 1.2
)

In [ ]:
model.save_pretrained_gguf("squat_coach_model_personality", tokenizer, quantization_method = "q4_k_m")

In [ ]:
from google.colab import files

# This is the exact name from your 'Generated files' log
files.download('Llama-3.2-3B-Instruct.Q4_K_M.gguf')